In [8]:
# 데이터를 분산 처리하고 Spark SQL/입출력을 사용하기 위한 핵심 실행 컨텍스트를 생성합니다.
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [9]:
# Parquet 데이터를 로드해 컬럼형 포맷 기반으로 효율적으로 분석합니다.
df_green = spark.read.parquet('data/pq/green/*/*')

In [12]:
# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 이 셀은 다음 분석 단계를 위한 중간 결과를 계산합니다.
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [13]:
# Parquet 데이터를 로드해 컬럼형 포맷 기반으로 효율적으로 분석합니다.
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [14]:
# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 이 셀은 다음 분석 단계를 위한 중간 결과를 계산합니다.
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [15]:
# green/yellow 공통 컬럼을 추려 유니온 가능한 스키마를 맞춥니다.
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [16]:
# 컬럼 연산과 집계를 표현하기 위해 Spark SQL 함수를 임포트합니다.
from pyspark.sql import functions as F

In [17]:
# green/yellow 공통 컬럼을 추려 유니온 가능한 스키마를 맞춥니다.
df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

In [18]:
# green/yellow 공통 컬럼을 추려 유니온 가능한 스키마를 맞춥니다.
df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

In [19]:
# 동일 스키마의 두 데이터셋을 통합해 단일 분석 대상으로 만듭니다.
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [20]:
# 결과 샘플을 눈으로 확인해 변환 로직이 맞는지 검증합니다.
df_trips_data.groupBy('service_type').count().show()

+------------+-------+
|service_type|  count|
+------------+-------+
|       green|1734051|
|      yellow|6405008|
+------------+-------+



In [21]:

# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 컬럼 목록을 확인해 조인/유니온 전후 스키마 정합성을 점검합니다.
df_trips_data.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'service_type']

In [23]:
# DataFrame을 임시 테이블로 등록해 Spark SQL로 조회할 수 있게 합니다.
df_trips_data.registerTempTable('trips_data')

In [24]:
# 결과 샘플을 눈으로 확인해 변환 로직이 맞는지 검증합니다.
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 1734051|
|      yellow| 6405008|
+------------+--------+



In [25]:
# SQL 쿼리로 집계/분석 로직을 실행해 결과 DataFrame을 생성합니다.
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [26]:
# 후속 분석 성능을 위해 결과를 Parquet 포맷으로 저장합니다.
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')